In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
BASE_PATH = '/content/drive/MyDrive/datas_for_research_vicious_circle_project'
os.chdir(BASE_PATH)
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.linear_model import LinearRegression






In [ ]:
def load_data(paths):
    return {
        "benefits": pd.read_excel(paths["benefits"], header = None),
        "lamas": pd.read_excel(paths["lamas"], sheet_name="נתונים פיזיים ונתוני אוכלוסייה ", header = None),
        "socio_regional": pd.read_excel(paths["socio_regional"], header = None),
        "periph_regional": pd.read_excel(paths["periph_regional"], header = None),
        "coordinates": pd.read_csv(paths["coordinates"])
    }

In [ ]:
def merge_lamas(df_benefits, df_lamas):
    before = len(df_benefits)

    df = df_benefits.merge(
        df_lamas[[
            "settlement_symbol",
            "socio_economic_index_cluster",
            "socio_economic_index_score",
            "peripherality_index_cluster",
            "peripherality_index_score"
        ]],
        on="settlement_symbol",
        how="left"
    )

    assert len(df) == before, (
        "❌ Lost rows in LAMAS merge"
        f"before={before}, after={len(df)}"
    )
  #  assert len(df) == before, "❌ Lost rows in LAMAS merge"
    return df

In [ ]:
def merge_index_from_regional(df_main, df_regional, index_cols, key="settlement_symbol"):
    """
    Completes missing index values in df_main using df_regional,
    without overwriting existing data. Ensures no rows are lost or added during merge.
    """
    before = len(df_main)

    # מיזוג
    df = df_main.merge(
        df_regional[[key] + index_cols],
        on=key,
        how="left",
        suffixes=("", "_reg")
    )
    # בדיקת בטיחות: לא נוספו / לא נעלמו שורות
    assert len(df) == before, (
        f"❌ Row count mismatch after merge on '{key}': "
        f"before={before}, after={len(df)}"
    )

    for col in index_cols:
        df[col] = df[col].combine_first(df[f"{col}_reg"])

    # ניקוי עמודות עזר
    df = df.drop(columns=[f"{col}_reg" for col in index_cols])

    return df

In [ ]:
def merge_coordinates(df_main, df_coordinates):
    before = len(df_main)
    df = df_main.merge(
        df_coordinates[[
            'settlement_code',
            'lat',
            'lon'
        ]],
        left_on="settlement_symbol",
        right_on="settlement_code",
        how="left"
    )
    df = df.drop(columns=["settlement_code"])

    assert len(df) == before, (
        "❌ Lost rows in COORDINATES merge"
        f"before={before}, after={len(df)}"
    )
    return df

In [ ]:
def clean_values(df):
    socio_economic_peripherality_cols = df.columns[-6:-2]
    df = df.dropna(subset=socio_economic_peripherality_cols, how="all")
    # 3. הסרת עמודות לא רצויות
    cols_to_drop = [
        "settlement_type",
        "injury_allowance",
        "recipients_of_the_senior_citizen_pension_only",
        "recipients_of_the_pension_with_income_supplementation",
        "total_recipients_of_old_age_and/or_survivors’_benefits",
        "num_families_receiving_child_benefit",
        "num_children_receiving_child_benefit",
        "families_with_4+_children_receiving_child_benefit",
        "maternity_benefits",
        "alimony"
    ]

    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # 4. טיפוסי נתונים
    categorial_cols = ["socio_economic_index_cluster","peripherality_index_cluster"]
    numeric_cols = df.loc[:, 'total_population':'unemployment_benefit'].columns
    float_cols = ["socio_economic_index_score","peripherality_index_score", "lon", "lat"]

    for col in df.columns:
        if col in categorial_cols:
          df[col] = df[col].astype("category")
        elif col in numeric_cols:
          df[col] = df[col].astype(str).replace({r'\*\*\*': '5', r'\.\.': '5'}, regex=True).str.replace(",", "", regex=False)
          df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif col in float_cols:
          df[col] = df[col].astype(float)
        else:
          df[col] = df[col].astype("object")
    return df

In [ ]:
def save_dataset(df, name, output_dir="data/processed"):
    """
    Save DataFrame both as CSV and Pickle.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{name}.csv"
    pkl_path = output_dir / f"{name}.pkl"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_pickle(pkl_path)

    print(f"✅ Saved CSV: {csv_path}")
    print(f"✅ Saved Pickle: {pkl_path}")


In [ ]:
paths = {
    "benefits": "benefits_2024_12.xlsx",
    "lamas": "p_libud_23.xlsx",
    "socio_regional": "24_24_230t3.xlsx",
    "periph_regional": "24_22_420t3.xlsx",
    "coordinates": "israel_settlements_all_with_coords.csv"
}

dfs = load_data(paths)

df_benefits = dfs["benefits"].iloc[5:].copy()
df_benefits = df_benefits.reset_index(drop=True)
df_benefits.columns = [
    "settlement_name",
    "settlement_symbol",
    "settlement_type",
    "total_population",
    "population_0_17",
    "population_18_64",
    "population_65_plus",
    "total_recipients_of_old_age_and/or_survivors’_benefits",
    "recipients_of_the_pension_with_income_supplementation",
    "recipients_of_the_senior_citizen_pension_only",
    "long_term_care_benefit",
    "general_disability_benefit",
    "special_services_for_persons_with_disabilities",
    "disabled_child_benefit",
    "mobility_benefit",
    "work_injury_victims_receiving_disability_and_dependents’_benefits",
    "injury_allowance",
    "num_families_receiving_child_benefit",
    "num_children_receiving_child_benefit",
    "families_with_4+_children_receiving_child_benefit",
    "maternity_benefits",
    "alimony",
    "income_support_benefit",
    "unemployment_benefit"
]

df_lamas = dfs["lamas"].iloc[9:].copy()
df_lamas = df_lamas.reset_index(drop=True)
df_lamas = df_lamas[df_lamas[3] != 'מועצה אזורית']
df_lamas.rename(columns={1: "settlement_symbol", 250: "socio_economic_index_cluster",
                         251: "socio_economic_index_score",256: "peripherality_index_cluster",
                         257: "peripherality_index_score"}, inplace=True)

df_socio = dfs["socio_regional"].iloc[10:].copy()
df_socio = df_socio.reset_index(drop=True)
df_socio = df_socio.iloc[:-8]
df_socio.rename(columns={5: "settlement_symbol", 12: "socio_economic_index_cluster", 10: "socio_economic_index_score"}, inplace=True)

#DROP_ON_TOP=9
df_periph = dfs["periph_regional"].iloc[9:].copy()
#NOTES_IN_THE_END=4
df_periph = df_periph.reset_index(drop=True)
df_periph = df_periph.iloc[:-4]
df_periph.rename(columns={4: "settlement_symbol", 12: "peripherality_index_cluster", 10: "peripherality_index_score"}, inplace=True)

data_master = merge_lamas(df_benefits, df_lamas)
data_master = merge_index_from_regional(data_master, df_socio,
    index_cols=[
        "socio_economic_index_cluster",
        "socio_economic_index_score"
    ]
)
data_master = merge_index_from_regional(data_master, df_periph,
    index_cols=[
        "peripherality_index_cluster",
        "peripherality_index_score"
    ]
)
data_master = merge_coordinates(data_master, dfs["coordinates"])
data_master = clean_values(data_master)
save_dataset(data_master, "benefits_final")

In [ ]:
data_master

In [ ]:
data_master['general_disability_rate'] = round((data_master['general_disability_benefit']/data_master['population_18_64']) * 100,2)
data_master['special_services_disability_rate']= round((data_master['special_services_for_persons_with_disabilities']/data_master['population_18_64']) * 100,2)
data_master['mobility_disability_rate'] = round((data_master['mobility_benefit']/data_master['population_18_64']) * 100,2)
data_master['income_support_rate'] = round((data_master['income_support_benefit']/data_master['population_18_64']) * 100,2)
data_master['long_term_care_rate'] = round((data_master['long_term_care_benefit']/data_master['population_65_plus']) * 100,2)
data_master['unemployment_rate'] = round((data_master['unemployment_benefit']/data_master['population_18_64']) * 100,2)
data_master['work_injury_victims_rate'] = round((data_master['work_injury_victims_receiving_disability_and_dependents’_benefits']/data_master['population_18_64']) * 100,2)
data_master['disabled_child_benefit_rate'] = round((data_master['disabled_child_benefit']/data_master['population_0_17']) * 100,2)


In [ ]:
# ---------- דאטה ----------
df = data_master.dropna(
    subset=[
        "lat", "lon", "settlement_name",
        "total_population",
        "socio_economic_index_cluster",
        "general_disability_rate",
        "unemployment_rate",
    ]
).copy()

df["cluster"] = df["socio_economic_index_cluster"].astype(int)
df["disability"] = df["general_disability_rate"].astype(float)
df["unemployment"] = df["unemployment_rate"].astype(float)
df["population"] = df["total_population"].astype(int)

cluster_norm = ((df["cluster"] - 1) / 9).to_numpy()
disability_arr = df["disability"].to_numpy()
unemployment_arr = df["unemployment"].to_numpy()

# ---------- customdata ----------
# סדר: [cluster, disability, unemployment, population]
customdata_arr = df[["cluster", "disability", "unemployment", "population"]].to_numpy()

# ---------- hover templates ----------
hover_cluster = (
    "<b>%{text}</b><br>"
    "Socio-Economic Cluster: %{customdata[0]}<br>"
    "Population: %{customdata[3]:,}"
    "<extra></extra>"
)

hover_disability = (
    "<b>%{text}</b><br>"
    "General Disability Rate: %{customdata[1]:.2f}%<br>"
    "Population: %{customdata[3]:,}"
    "<extra></extra>"
)

hover_unemployment = (
    "<b>%{text}</b><br>"
    "Unemployment Rate: %{customdata[2]:.2f}%<br>"
    "Population: %{customdata[3]:,}"
    "<extra></extra>"
)

# ---------- colorscale דיסקרטי לקלאסטרים ----------
green_gold_red = [
    [0.00, "#d73027"],
    [0.50, "#c9DD00"],
    [1.00, "#1a9850"],
]
cluster_colors = px.colors.sample_colorscale(green_gold_red, np.linspace(0, 1, 10))

cluster_scale = []
for i, c in enumerate(cluster_colors):
    t = i / 9
    cluster_scale.append([t, c])
    cluster_scale.append([t, c])

# ---------- colorscales רציפים ----------
scale_disability = [
    [0.00, "#1a9850"],
    [0.45, "#fee08b"],
    [0.75, "#d73027"],
    [1.00, "#000000"],
]
scale_unemployment = "Viridis"

# ---------- size scaling ----------
size_max = 38
vmax = float(df["population"].max())
sizeref = 2.0 * vmax / (size_max ** 2) if vmax > 0 else 1

# ---------- trace ----------
fig = go.Figure(
    go.Scattermapbox(
        lat=df["lat"],
        lon=df["lon"],
        mode="markers",
        text=df["settlement_name"],
        customdata=customdata_arr,
        hovertemplate=hover_cluster,
        marker=dict(
            size=df["population"],
            sizemode="area",
            sizeref=sizeref,
            opacity=0.9,

            # ברירת מחדל: קלאסטר
            color=cluster_norm,
            colorscale=cluster_scale,
            cmin=0,
            cmax=1,
            showscale=True,
            colorbar=dict(
                title="Cluster (1–10)",
                tickmode="array",
                tickvals=[i / 9 for i in range(10)],
                ticktext=[str(i) for i in range(1, 11)],
                len=0.85,
                x=1.02,
                xanchor="left",
            ),
        ),
    )
)

# ---------- layout ----------
fig.update_layout(
    mapbox=dict(
        style="carto-positron",
        center=dict(lat=31.3, lon=34.9),
        zoom=6.8,
    ),
    width=500,
    height=900,
    margin=dict(l=0, r=140, t=20, b=0),
)

# ---------- Dropdown ----------
fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.0, y=1.0,
            xanchor="left", yanchor="top",
            showactive=True,

            # 👇 זה העיקר:
            font=dict(size=14),                 # הגדלת כתב של התפריט
            pad=dict(t=10, b=10, r=10, l=10),   # יותר "בשר" לכפתור
            buttons=[
                dict(
                    label="Socio-Economic Cluster",
                    method="restyle",
                    args=[{
                        "marker.color": [cluster_norm],
                        "marker.colorscale": [cluster_scale],
                        "marker.cmin": [0],
                        "marker.cmax": [1],
                        "marker.showscale": [True],
                        "marker.colorbar.title": ["Cluster (1–10)"],
                        "marker.colorbar.tickmode": ["array"],
                        "marker.colorbar.tickvals": [[i / 9 for i in range(10)]],
                        "marker.colorbar.ticktext": [[str(i) for i in range(1, 11)]],
                        "marker.colorbar.x": [1.02],
                        "hovertemplate": [hover_cluster],
                    }],
                ),
                dict(
                    label="General Disability Rate",
                    method="restyle",
                    args=[{
                        "marker.color": [disability_arr],
                        "marker.colorscale": [scale_disability],
                        "marker.cmin": [float(np.nanmin(disability_arr))],
                        "marker.cmax": [float(np.nanmax(disability_arr))],
                        "marker.showscale": [True],
                        "marker.colorbar.title": ["General Disability Rate (%)"],
                        "marker.colorbar.tickmode": ["auto"],
                        "marker.colorbar.x": [1.02],
                        "hovertemplate": [hover_disability],
                    }],
                ),
                dict(
                    label="Unemployment Rate",
                    method="restyle",
                    args=[{
                        "marker.color": [unemployment_arr],
                        "marker.colorscale": [scale_unemployment],
                        "marker.cmin": [float(np.nanmin(unemployment_arr))],
                        "marker.cmax": [float(np.nanmax(unemployment_arr))],
                        "marker.showscale": [True],
                        "marker.colorbar.title": ["Unemployment Rate (%)"],
                        "marker.colorbar.tickmode": ["auto"],
                        "marker.colorbar.x": [1.02],
                        "hovertemplate": [hover_unemployment],
                    }],
                ),
            ],
        )
    ]
)

fig.show()

In [ ]:
social_cols = [
    'socio_economic_index_score',
    'peripherality_index_score',
    'income_support_rate'
]

df_social = data_master[social_cols].copy()

# היפוך כיוון למדדים "חיוביים"
df_social['socio_economic_index_score'] *= -1
df_social['peripherality_index_score'] *= -1


data_master['social_index'] = StandardScaler().fit_transform(df_social).mean(axis=1)


In [ ]:
health_cols = [
    'general_disability_rate',
    'special_services_disability_rate',
    'mobility_disability_rate'
]

data_master['health_index'] = StandardScaler().fit_transform(data_master[health_cols]).mean(axis=1)


In [ ]:
corr_val = data_master['social_index'].corr(data_master['health_index'], method='spearman')
print(f"✅ Final Correlation (Simple Index): {corr_val:.3f}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. הכנת הנתונים
df_calc = data_master.copy()

# היפוך כיוון למדדים של הלמ"ס (כדי שגבוה = מצוקה)
df_calc['socio_economic_index_score'] *= -1
df_calc['peripherality_index_score'] *= -1

social_cols = [
    'socio_economic_index_score',
    'peripherality_index_score',
    'income_support_rate'
]

health_cols = [
    'general_disability_rate',
    'special_services_disability_rate',
    'mobility_disability_rate'
]

cols_for_pca = social_cols + health_cols
df_pca_clean = df_calc.dropna(subset=cols_for_pca).copy()

print(f"Number of localities for PCA analysis: {len(df_pca_clean)}")

# 2. נרמול הנתונים (על הקבוצה המשותפת בלבד)
X_health = StandardScaler().fit_transform(df_pca_clean[health_cols])
X_social = StandardScaler().fit_transform(df_pca_clean[social_cols])

# 3. חישוב PCA - מדד בריאות
pca_h = PCA(n_components=1)
health_pca_vals = pca_h.fit_transform(X_health)

# בדיקת כיוון ותיקון (מול העמודה הראשונה בטבלה המנוקה)
if np.corrcoef(health_pca_vals.T, df_pca_clean[health_cols[0]])[0, 1] < 0:
    health_pca_vals = health_pca_vals * -1

# 4. חישוב PCA - מדד חברתי
pca_s = PCA(n_components=1)
social_pca_vals = pca_s.fit_transform(X_social)

# בדיקת כיוון ותיקון
if np.corrcoef(social_pca_vals.T, df_pca_clean['income_support_rate'])[0, 1] < 0:
    social_pca_vals = social_pca_vals * -1

# 5. שמירה בטבלה המקורית
# אנו משתמשים ב-Index כדי להחזיר את הערכים למקומות הנכונים ב-data_master
data_master.loc[df_pca_clean.index, 'health_index_pca'] = health_pca_vals
data_master.loc[df_pca_clean.index, 'social_index_pca'] = social_pca_vals

# 6. בדיקת הקורלציה הסופית
# משתמשים ב-df_pca_clean כדי לוודא שאין NaN בחישוב הקורלציה
corr_pca = np.corrcoef(social_pca_vals.flatten(), health_pca_vals.flatten())[0, 1]
# או בשיטת פנדס (ספירמן):
spearman_corr = data_master.loc[df_pca_clean.index, 'social_index_pca'].corr(
    data_master.loc[df_pca_clean.index, 'health_index_pca'], method='spearman'
)

print(f"🚀 Final Correlation (PCA Method, Spearman): {spearman_corr:.3f}")

In [ ]:
df = data_master.copy()

cols = [
    'social_index_pca',
    'health_index_pca',
    'socio_economic_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols)

fig = px.scatter(
    df,
    x='social_index_pca',
    y='health_index_pca',
    hover_name='settlement_name',
    size='total_population',
    size_max = 40,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis',
    trendline='ols',
    labels={
        'social_index_pca': 'Social-Economic Vulnerability Index',
        'health_index_pca': 'Disability Health Index',
        'total_population': 'Population size',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Correlation Between Social-Economic Vulnerability and Disability'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),   # או 'black'
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between the Social Index and the Health Index is {df['social_index_pca'].corr(df['health_index_pca'], method= 'spearman'):.2f}")

In [ ]:
#from sklearn.linear_model import LinearRegression
df = data_master.copy()

cols = [
    'social_index',
    'health_index'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols)
# 1. הכנת הנתונים לרגרסיה
# אנו מנבאים את מדד הבריאות (Y) על בסיס המדד החברתי (X)
X = df[['social_index']]
y = df['health_index']

model = LinearRegression().fit(X, y)

# 2. חישוב השאריות (Residuals)
# השארית היא המרחק בין המציאות לבין המודל
df['predicted_health'] = model.predict(X)
df['residual'] = df['health_index'] - df['predicted_health']

# 3. יצירת הגרף המשודרג
fig = px.scatter(
    df,
    x='social_index',  # ציר ה-X: המצב החברתי (הגורם)
    y='health_index',  # ציר ה-Y: המצב הבריאותי (התוצאה)
    hover_name='settlement_name',
    hover_data={
        'residual': ':.2f',
        'socio_economic_index_cluster': True,
        'social_index': False, # מסתירים כי זה כבר בציר
        'health_index': False  # מסתירים כי זה כבר בציר
    },
    size='total_population',
    size_max=40,

    # --- צביעה לפי חריגות ---
    color='residual',
    # RdBu_r: Red-Blue reversed.
    # כחול (שלילי) = נכות נמוכה מהצפוי (חוסן/טוב)
    # אדום (חיובי) = נכות גבוהה מהצפוי (מצוקה/רע)
    color_continuous_scale='RdBu_r',

    # הוספת קו המגמה השחור
    trendline='ols',
    trendline_color_override='black',

    title='Resilience vs. Distress: Deviation from Expected Disability Rate',
    labels={
        'social_index': 'Social Vulnerability (High = Poor/Peripheral)',
        'health_index': 'Disability Severity (High = More Disabled)',
        'residual': 'Deviation (Severity)'
    }
)

# 4. עיצוב סופי
fig.update_layout(
    title_font_size=20,
    coloraxis_colorbar=dict(
        title="Anomaly Type",
        tickvals=[-1, 0, 1],
        ticktext=["Resilient\n(Better than expected)", "Normal", "Distress\n(Worse than expected)"]
    )
)

fig.show()

# 5. הדפסת התובנות (הערים הכי מעניינות)
print("--- Top 5 'Resilient' Cities (Blue) ---")
print("Cities with POOR social status but GOOD health outcomes:")
#print(df.sort_values('residual').head(10)[['settlement_name', 'socio_economic_index_cluster', 'residual']])
print(df[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']].sort_values(by = 'residual').head(10))

print("\n--- Top 5 'Distress' Cities (Red) ---")
print("Cities with BETTER social status but BAD health outcomes:")
print(df.sort_values('residual', ascending=False).head(10)[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']])

In [ ]:
df = data_master.copy()

cols = [
    'general_disability_rate',
    'income_support_rate',
    'socio_economic_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols] = df[cols].astype(float)
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='general_disability_rate',
    y='income_support_rate',
    hover_name='settlement_name',
  #  size='total_population',
  #  size_max= 50,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis',
    trendline='ols',
    labels={
        'general_disability_rate': 'General Disability Rate (%)',
        'income_support_rate': 'Income Support Rate (%)',
        'total_population': 'Population size',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Correlation Between General Disability Rate and Income Support Rate'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between general disability rate and income support rate  is {data_master['general_disability_rate'].corr(data_master['income_support_rate'], method= 'spearman'):.2f}")

In [ ]:
df = data_master.copy()

cols = [
    'general_disability_rate',
    'socio_economic_index_score',
    'peripherality_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='general_disability_rate',
    y='socio_economic_index_score',
    hover_name='settlement_name',
    size='total_population',
    size_max=40,
    color='peripherality_index_cluster',
    color_continuous_scale='Plasma',
    trendline='ols',
    labels={
        'general_disability_rate': 'General Disability Rate (%)',
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'total_population': 'Population size',
        'peripherality_index_cluster': 'Peripherality Index'
    },
    title='Correlation Between General Disability Rate and Socio-Economic Index'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between general disability rate and socio economic index  is {data_master['general_disability_rate'].corr(data_master['socio_economic_index_score'], method= 'spearman'):.2f}")

In [ ]:
#from sklearn.linear_model import LinearRegression
df = data_master.copy()

cols = [
    'general_disability_rate',
    'socio_economic_index_score'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols)
# 1. הכנת הנתונים לרגרסיה
X = df[['general_disability_rate']]
y = df['socio_economic_index_score']

model = LinearRegression().fit(X, y)

# 2. חישוב השאריות (Residuals)
# השארית היא המרחק בין המציאות לבין המודל
df['predicted_socio_economic_index'] = model.predict(X)
df['residual'] = df['socio_economic_index_score'] - df['predicted_socio_economic_index']

# 3. יצירת הגרף המשודרג
fig = px.scatter(
    df,
    x='general_disability_rate',
    y='socio_economic_index_score',
    hover_name='settlement_name',
    hover_data={
        'residual': ':.2f',
        'socio_economic_index_cluster': True,
        'general_disability_rate': False, # מסתירים כי זה כבר בציר
        'socio_economic_index_score': False  # מסתירים כי זה כבר בציר
    },
    size='total_population',
    size_max=40,

    # --- צביעה לפי חריגות ---
    color='residual',
    # RdBu_r: Red-Blue reversed.

    color_continuous_scale='RdBu_r',

    # הוספת קו המגמה השחור
    trendline='ols',
    trendline_color_override='black',

    title='Resilience vs. Distress: Deviation from Expected Disability Rate',
    labels={
        'general_disability_rate': 'General Disability Rate (%)',
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'residual': 'Deviation (Severity)'
    }
)

# 4. עיצוב סופי
fig.update_layout(
    title_font_size=20,
    coloraxis_colorbar=dict(
        title="Anomaly Type",
        tickvals=[-4, 0, 4],
        ticktext=["Resilient\n(Better than expected)", "Normal", "Distress\n(Worse than expected)"]
    )
)

fig.show()

# 5. הדפסת התובנות (הערים הכי מעניינות)
print("--- Top 5 'Resilient' Cities (Blue) ---")
print("Cities with POOR social status but GOOD health outcomes:")
#print(df.sort_values('residual').head(10)[['settlement_name', 'socio_economic_index_cluster', 'residual']])
print(df[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']].sort_values(by = 'residual').head(10))

print("\n--- Top 5 'Distress' Cities (Red) ---")
print("Cities with BETTER social status but BAD health outcomes:")
print(df.sort_values('residual', ascending=False).head(10)[['settlement_name', 'socio_economic_index_cluster', 'residual', 'total_population']])


In [ ]:
df = data_master.copy()

cols = [
    'long_term_care_rate',
    'socio_economic_index_score',
    'peripherality_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='long_term_care_rate',
    y='socio_economic_index_score',
    hover_name='settlement_name',
    size='total_population',
    size_max=40,
    color='peripherality_index_cluster',
    color_continuous_scale='Plasma',
    trendline='ols',
    labels={
        'long_term_care_rate': 'Long Term care Rate (%)',
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'total_population': 'Population size',
        'peripherality_index_cluster': 'Peripherality Index'
    },
    title='Correlation Between Long Term care Rate and Socio-Economic Index'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between Long Term care Rate and socio economic index  is {data_master['long_term_care_rate'].corr(data_master['socio_economic_index_score'], method= 'spearman'):.2f}")

In [ ]:
df = data_master.copy()

cols = [
    'long_term_care_rate',
    'income_support_rate',
    'peripherality_index_cluster'
]

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols] = df[cols].astype(float)
df = df.dropna(subset= cols)

fig = px.scatter(
    df,
    x='long_term_care_rate',
    y='income_support_rate',
    hover_name='settlement_name',
    size='total_population',
    size_max=40,
    color='peripherality_index_cluster',
    color_continuous_scale='Plasma',
    trendline='ols',
    labels={
        'long_term_care_rate': 'Long Term care Rate (%)',
        'income_support_rate': 'Income Support Rate (%)',
        'total_population': 'Population size',
        'peripherality_index_cluster': 'Peripherality Index'
    },
    title='Correlation Between Long Term care Rate and Income Support Rate'
)

fig.update_traces(
    marker=dict(opacity=0.7),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color='darkblue', width=3),
    selector=dict(mode='lines')
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()
print(f"The correlation between Long Term care Rate and Income Support Rate is {data_master['long_term_care_rate'].corr(data_master['income_support_rate'], method= 'spearman'):.2f}")

In [ ]:
df = data_master.copy()
#df = df.drop(index=[2, 39,27, 38, 195,158,218,269,198,88,17,191,196,154,53,117,116])
corr = df[social_cols + health_cols].corr(method="spearman")
focused_corr = corr.loc[health_cols, social_cols]


fig = px.imshow(
    focused_corr,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title='Spearman Correlations: Social vs Health Indicators'
)

fig.update_layout(
    xaxis_title='Social / Economic Indicators',
    yaxis_title='Health / Disability Indicators',
    title_font_size=18
)

fig.show()

The Intergenerational Trap:

In [ ]:
# --- הכנה: חלוקה גיאוגרפית גסה (צפון, מרכז, דרום) ---
# נשתמש בקווי רוחב (Latitude) כדי לחלק את הארץ
def get_region(lat):
    if pd.isna(lat): return 'Unknown'
    if lat > 32.4: return 'North' # צפונית לחדרה בערך
    if lat < 31.7: return 'South' # דרומית לאשדוד/גדרה
    return 'Center'

data_master['region'] = data_master['lat'].apply(get_region)

# ==============================================================================
# תובנה 1: מלכודת הדורות (Intergenerational Trap)
# ==============================================================================
# סינון: רק יישובים עם אוכלוסייה משמעותית כדי למנוע רעש
df_trap = data_master.copy()
national_avg_adult_disability = (df_trap['general_disability_benefit'].sum() / df_trap['population_18_64'].sum()) * 100
national_avg_child_disability = (df_trap['disabled_child_benefit'].sum() / df_trap['population_0_17'].sum()) * 100
df_trap['socio_economic_index_cluster'] = pd.to_numeric(df_trap['socio_economic_index_cluster'], errors='coerce')

fig_trap = px.scatter(
    df_trap,
    x='general_disability_rate',
    y='disabled_child_benefit_rate',
    hover_name='settlement_name',
  #  size='total_population',
    color='socio_economic_index_cluster',
    color_continuous_scale='RdYlGn', # ירוק=טוב (אשכול גבוה), אדום=רע
    title='The Intergenerational Trap: Child vs. Adult Disability',
    labels={'general_disability_rate': 'Disability Benefit Recipient Rate (Ages 18-64) %',
            'disabled_child_benefit_rate': 'Child Disability Rate (Ages 0-17) %',
            'socio_economic_index_cluster': 'Socio-Economic Cluster'}
)

fig_trap.update_layout(
    shapes=[
        # קו אנכי: הממוצע הארצי של נכות מבוגרים
        dict(
            type="line",
            x0=national_avg_adult_disability,
            x1=national_avg_adult_disability,
            y0=0,
            y1=df_trap['disabled_child_benefit_rate'].max(),
            line=dict(color="black", dash="dash", width=2)
        ),

        # קו אופקי: הממוצע הארצי של נכות ילדים
        dict(
            type="line",
            x0=0,
            x1=df_trap['general_disability_rate'].max(),
            y0=national_avg_child_disability,
            y1=national_avg_child_disability,
            line=dict(color="black", dash="dash", width=2)
        ),

        # ריבוע אדום: אזור הסכנה (מעל הממוצע בשני המדדים)
        dict(
            type="rect",
            x0=national_avg_adult_disability,
            y0=national_avg_child_disability,
            x1=df_trap['general_disability_rate'].max(),
            y1=df_trap['disabled_child_benefit_rate'].max(),
            line=dict(color="Red", width=0), # ללא מסגרת
            fillcolor="Red",
            opacity=0.1,
            layer="below" # שהריבוע יהיה מתחת לנקודות
        )
    ]
)

# הוספת טקסט הסבר על הקווים
fig_trap.add_annotation(
    x=national_avg_adult_disability,
    y=df_trap['disabled_child_benefit_rate'].max(),
    text="National Avg (Adults)",
    showarrow=False,
    yshift=10
)

fig_trap.show()

print("--- 🚨 Red Alert: Towns with High Child AND Adult Disability ---")
high_child = df_trap['disabled_child_benefit_rate'] > df_trap['disabled_child_benefit_rate'].quantile(0.75)
high_adult = df_trap['general_disability_rate'] > df_trap['general_disability_rate'].quantile(0.75)
df_trap[high_child & high_adult][['settlement_name', 'region', 'socio_economic_index_cluster', 'disabled_child_benefit_rate', 'general_disability_rate']].sort_values('disabled_child_benefit_rate', ascending=False).head(10)

**The Data Story:**
This scatter plot reveals a disturbing correlation between **working-age disability** (parents) and **child disability** (future generation). The "Red Zone" (Top-Right Quadrant) represents localities where both rates exceed the national weighted average.

**Key Findings for Decision Makers:**
*   **The Vicious Cycle:** Disability in these areas is not an isolated individual event but a **household phenomenon**. Poverty and health issues are being transferred from generation to generation.
*   **Target Audience:** The localities in the top-right quadrant (marked by the red square) are the highest priority for intervention.
*   **Action Item:** Standard employment programs will likely fail here. These families require **holistic, multi-generational intervention** (treating the parent and child as a single unit) to break the cycle of dependency.

If there is a limited budget, it's better to put it on the 10 cities on this list.


<div class="alert alert-warning">
<h3>⚠️ Methodological Note: Administrative Data vs. Reality</h3>

**Critical Context for Decision Makers:**
The analysis below uses <b>Bituach Leumi administrative data</b>. It is crucial to distinguish between "Benefit Recipients" and the theoretical status of an individual:

*   <b>Unemployment Data:</b> Represents only those currently receiving <b>short-term unemployment benefits</b>. Long-term unemployed individuals who have exhausted their eligibility (common in distressed localities) are <b>NOT</b> counted in this specific denominator.
*   <b>Disability Data:</b> Represents those who successfully claimed benefits.

<b>Implication:</b> In the following graph ("The Welfare Trap"), a high ratio often indicates <b>structural, long-term unemployment</b> where residents have shifted from the temporary "unemployment track" to the permanent "disability track".
</div>

In [ ]:
# ==============================================================================
#   (Chronic versus temporary benefits)תובנה 2: מסלול כרוני מול מסלול זמני
# ==============================================================================
df = data_master.copy()
# יחס נכות לאבטלה: כמה נכים יש על כל מובטל?
# יחס גבוה = ייתכן שאנשים "בורחים" לנכות כי אין עבודה, או שיש בעיה מבנית עמוקה
df['chronic_vs_temp_ratio'] = df['general_disability_benefit'] / df['unemployment_benefit']

# סינון רעשים  עם 0 מקבלי אבטלה)
df_ratio = df[df['unemployment_benefit'] > 10]

fig_ratio = px.bar(
    df_ratio.sort_values('chronic_vs_temp_ratio', ascending=False).head(15),
    x='settlement_name',
    y='chronic_vs_temp_ratio',
  #  color='region',
    title='The Welfare Trap: Ratio of Chronic (General Disability) to Temporary (Unemployment) Benefits',
    labels={'chronic_vs_temp_ratio': 'Ratio: Disability Recipients / Unemployment Recipients', 'settlement_name': 'Locality'},
    text_auto='.1f'
)
fig_ratio.update_layout(
    xaxis_tickangle=-45,
    # הוספת הערה בתחתית הגרף שמסבירה את המגבלה
    annotations=[dict(
        text="*Note: High ratio may indicate long-term unemployment where benefits have been exhausted.",
        x=0, y=-0.25, xref="paper", yref="paper", showarrow=False, font=dict(size=10, color="gray")
    )]
)
fig_ratio.show()


Localities with a high Disability-to-Unemployment ratio indicate a **collapsed labor market**. In these areas, disability benefits have become the de-facto social safety net instead of employment.

**Strategic Recommendations:**
1.  **Shift from Placement to Rehabilitation:** Standard employment bureaus are ineffective here. Resources should be diverted to **Vocational Rehabilitation Centers** that combine social work with soft-skills training.
2.  **"Laron Law" Optimization:** Launch targeted campaigns in these specific towns to educate residents on their right to work *without* losing their entire benefit (incentivizing part-time work).
3.  **Supply-Side Intervention:** Subsidies in these zones should focus exclusively on **"Inclusive Employers"** willing to adapt roles for individuals with chronic health issues.

---



In [ ]:
# ==============================================================================
# תובנה 3: השוואה אזורית (צפון מול דרום)
# ==============================================================================

# נשווה רק יישובים מאשכולות נמוכים (1-4) כדי להשוות "תפוחים לתפוחים"
df_low_socio = data_master[data_master['socio_economic_index_cluster'].astype(float) <= 4].copy()

fig_region = px.box(
    df_low_socio,
    x='region',
    y='health_index',
    color='region',
    points='all', # מציג גם את הנקודות עצמן
    hover_name='settlement_name',
    title='Disability Rates in Low Socio-Economic Towns (Clusters 1-4): North vs. South vs. Center',
    category_orders={"region": ["North", "Center", "South"]}
)
fig_region.show()

**The Data Story:**
This analysis compares "apples to apples" (only low socio-economic settlements, clusters 1-4). The results are striking: **Geography dictates destiny.**
*   **The North (Blue):** Shows the highest median disability rate (~8%) and the most extreme outliers.
*   **The Center (Red):** Demonstrates a "protective effect" with significantly lower rates (~5%), even for equally poor populations.

**Key Takeaway for Decision Makers:**
Poverty in the periphery is "more toxic" than poverty in the center. The lack of accessible healthcare and employment infrastructure in the North exacerbates medical conditions.

**Action Item:**
**Prioritize the Northern District.** When allocating limited resources for health-employment interventions, a "poor town" in the North should receive higher weighting than a "poor town" in the Center.

In [ ]:
#הכנה לגרף
## ---------- שמות עמודות ----------
DISABILITY_COUNT_COL = "general_disability_benefit"
WORKING_AGE_POP_COL = "population_18_64"
RATE_COL = "general_disability_rate"
SE_COL = "socio_economic_index_score"

## ---------- דאטה ----------
dfv = data_master.dropna(
    subset=[DISABILITY_COUNT_COL, WORKING_AGE_POP_COL, RATE_COL, SE_COL]
).copy()

dfv["disability_count"] = dfv[DISABILITY_COUNT_COL].astype(float)
dfv["pop_working_age"] = dfv[WORKING_AGE_POP_COL].astype(float)
dfv["disability_rate"] = dfv[RATE_COL].astype(float)
dfv["se_score"] = dfv[SE_COL].astype(float)

## ---------- רבעונים ----------
quartile_labels = ["Q1 (Lowest)", "Q2 (Low)", "Q3 (Medium)", "Q4 (Highest)"]
dfv["se_quartile"] = pd.qcut(dfv["se_score"], 4, labels=quartile_labels)

## ---------- עמודות: ממוצע שיעור נכות לרבעון (לא משוקלל) ----------
summary = (
    dfv.groupby("se_quartile", observed=True)
       .agg(
           avg_disability_rate=("disability_rate", "mean"),
           n=("disability_rate", "size")
       )
       .reset_index()
)

summary["se_quartile"] = pd.Categorical(
    summary["se_quartile"],
    categories=quartile_labels,
    ordered=True
)
summary = summary.sort_values("se_quartile")

## ---------- Benchmark: משוקלל לפי אוכלוסייה בגיל עבודה (18–64) ----------
overall_working_age_weighted = (
    dfv["disability_count"].sum() / dfv["pop_working_age"].sum()
) * 100

## ---------- צבעים ידניים לרבעונים ----------
color_map = {
    "Q1 (Lowest)": "#d73017",
    "Q2 (Low)": "#ff6e31",
    "Q3 (Medium)": "#dee00b",
    "Q4 (Highest)": "#1ac850",
}

# ---------- גרף (Plotly Express) ----------
fig = px.bar(
    summary,
    x="se_quartile",
    y="avg_disability_rate",
    color="se_quartile",
    color_discrete_map=color_map,
    text=summary["avg_disability_rate"].round(2),
    category_orders={"se_quartile": quartile_labels},
    labels={
        "se_quartile": "Socio-Economic Quartile",
        "avg_disability_rate": "Average Disability Rate (%)",
    },
    title="Disability Rates by Socio-Economic Quartile"
)

# hover עם מספר יישובים (גודל N)
fig.update_traces(
    textposition="outside",
    customdata=summary[["n"]].to_numpy(),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Avg disability rate (settlement mean): %{y:.2f}%<br>"
        "Number of settlements: %{customdata[0]}"
        "<extra></extra>"
    )
)

# ---------- עיצוב ----------
fig.update_layout(
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=650,
    bargap=0.3,
    yaxis=dict(
        range=[0, 10.1],
        showgrid=True,
        gridcolor="rgba(0,0,0,0.2)"
    ),
    xaxis=dict(showgrid=False),
    showlegend=False
)

# ---------- קו benchmark ----------
fig.add_hline(
    y=overall_working_age_weighted,
    line_width=3,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Overall disabilty rate: {overall_working_age_weighted:.2f}%",
    annotation_position="top left",
)

fig.show()

המחקר של גל:
סיכום המחקר: בחירות מתודולוגיות והיגיון אנליטי
מטרת המחקר

המחקר נועד לבחון את הטענה הרווחת בדבר קשר מעגלי בין עוני, פריפריאליות ומוגבלות, אך לא רק לשאול האם קיים קשר – אלא היכן הוא חזק במיוחד, ובאילו סוגי מוגבלות. המטרה היישומית הייתה לזהות מוקדים בהם ניתן למקד התערבות חברתית-טכנולוגית אפקטיבית.

בחירת יחידת הניתוח: יישוב

המחקר מתבצע ברמת היישוב, ולא הפרט, מתוך הנחה שמוגבלות היא תופעה:

מרחבית

מבנית

מושפעת מהקשר חברתי-כלכלי

בחירה זו מאפשרת:

שילוב מדדים סוציו-אקונומיים, גיאוגרפיים ובריאותיים

זיהוי דפוסים אזוריים וחריגים יישוביים

תרגום ממצאים למדיניות ולהקצאת משאבים

למה לא הסתפקתי במספר מקבלי קצבאות?

מספרים מוחלטים של מקבלי קצבאות מושפעים ישירות מגודל היישוב. לכן, שימוש בהם היה:

מטעה

מחזק יישובים גדולים באופן מלאכותי

מונע השוואה אמיתית בין יישובים

הבחירה המרכזית: חישוב שיעורי תחלואה (Rates)
איך חישבתי את השיעורים?

לכל סוג קצבה חושב שיעור כך:

שיעור=מספר מקבלי קצבהאוכלוסיית היישוב הרלוונטית×1,000
שיעור=
אוכלוסיית היישוב הרלוונטית
מספר מקבלי קצבה
	​

×1,000

(או לכלל האוכלוסייה, בהתאם לסוג הקצבה)

למה חישוב שיעור ולא מספר?

המעבר משימוש במספרים מוחלטים לשיעורים:

מנטרל את גודל היישוב

מאפשר השוואה בין יישובים קטנים וגדולים

חושף פערים מבניים אמיתיים

כלומר:

השאלה אינה “איפה יש יותר נכים”, אלא
“איפה הסיכון להיות נכה גבוה יותר”

בידול בין סוגי מוגבלות – החלטה קריטית

במקום להתייחס ל”נכות” כמקשה אחת, בוצעה הבחנה בין סוגי קצבאות, מתוך הנחה שלכל סוג:

מנגנון סיבתי שונה

קשר שונה לעוני

שלוש קטגוריות מרכזיות שנבחנו:
1. נפגעי עבודה

נבחרו כקבוצת ביקורת, משום שהם:

נובעים מאירוע תעסוקתי

אינם תוצאה של תהליך עוני מצטבר

צפויים להיות פחות תלויים במצב סוציו-אקונומי

2. קצבאות סיעוד (זקנה)

נכללו בניתוח נפרד משום שהן:

תלויות מאוד בגיל

מושפעות גם מגורמים ביולוגיים טבעיים

עלולות “לזהם” ניתוח של נכות הקשורה לעוני

3. נכות כללית “קשורה לעוני”

נבנתה מדד אגרגטיבי הכולל:

נכות כללית

שירותים מיוחדים

הבטחת הכנסה

והוחרגו ממנו:

נפגעי עבודה

קצבאות זקנה

כך נוצר מדד שמייצג מוגבלות בגיל העבודה, עם תלות כלכלית – כלומר נכות שיש לה פוטנציאל להנציח עוני.

למה חישוב נפרד לפי גיל?

הניתוח בודד קבוצות גיל (בעיקר 18–64) כדי:

למנוע הטיה של יישובים “זקנים”

להתמקד באוכלוסייה שבה נכות משפיעה ישירות על השתתפות בשוק העבודה

לחדד את הקשר בין נכות, תעסוקה ועוני

בחינת הקשר ל-SEI

לאחר חישוב שיעורי התחלואה:

נבדק הקשר בין כל שיעור לבין מדד סוציו-אקונומי (SEI)

הוצגו גרפים עם:

קו מגמה

גודל נקודה לפי אוכלוסייה

צביעה לפי אשכול SEI

חושבו מדדי קורלציה (Pearson, Spearman)

למה ניתוח חריגים?

מעבר לממוצע:

זוהו יישובים החורגים מהקשר הכללי

בוצע ניתוח איכותני של חריגים “טובים” ו”רעים”

המטרה:

להבין אילו מנגנונים מחזקים או שוברים את הקשר

לאתר מוקדי התערבות והעתקה (POC)

המסר המרכזי של המתודולוגיה

המחקר אינו בוחן “כמה נכים יש”, אלא
איפה, באיזה הקשר חברתי, ובאיזה סוג מוגבלות – הסיכון לנכות גבוה במיוחד.

הבחירה לעבוד עם שיעורים, בידול סוגי קצבאות, ובידוד גילי עבודה היא שמאפשרת לעבור מגרף יפה לניתוח שמסוגל להנחות פעולה.

In [ ]:
# ====== התאמות נדרשות: החלף שמות עמודות אם שונים בדאטה שלך ======
COL_TOTAL_POP = 'total_population'
COL_SEI = 'socio_economic_index_score'
COL_PERIPH = 'peripherality_index_score'
COL_GENERAL = 'general_disability_benefit'
COL_LONGTERM = 'long_term_care_benefit'
COL_WORK_INJ = "work_injury_victims_receiving_disability_and_dependents’_benefits"  # בדוק שם מדויק
COL_POP_0_17 = 'population_0_17'
COL_POP_18_64 = 'population_18_64'
COL_POP_65 = 'population_65_plus'
# ===================================================================

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px # Added import for plotly

# df = pd.read_csv(...)  # טען כאן את ה-DataFrame שלך
# ---------------------------------------------------------------------------------
# 1) הכנה ונירמול לשיעורים per1000
# ---------------------------------------------------------------------------------
def safe_numeric(df, col):
    return pd.to_numeric(df[col], errors='coerce')

# המרת עמודות למספרים (אם לא כבר)
for c in [COL_TOTAL_POP, COL_SEI, COL_PERIPH, COL_GENERAL, COL_LONGTERM, COL_WORK_INJ,
          COL_POP_0_17, COL_POP_18_64, COL_POP_65]:
    if c in df.columns:
        df[c] = safe_numeric(df, c)

# Drop שורות בלי אוכלוסיה תקינה
df = df[df[COL_TOTAL_POP].notna() & (df[COL_TOTAL_POP] > 0)].copy()

# מחשבים שיעורים per1000 לכל קצבה קיימת
benefit_cols = [COL_GENERAL, COL_LONGTERM, COL_WORK_INJ]
for c in benefit_cols:
    if c in df.columns:
        df[c + '_per1000'] = df[c] / df[COL_TOTAL_POP] * 1000

# שיעורים לפי גיל (אם יש נתונים)
if COL_POP_18_64 in df.columns:
    df['rate_18_64_per1000'] = (df[COL_GENERAL] / df[COL_POP_18_64]) * 1000
if COL_POP_65 in df.columns:
    df['rate_65_plus_per1000'] = (df[COL_GENERAL] / df[COL_POP_65]) * 1000

# ---------------------------------------------------------------------------------
# 2) שיעור אגרגטיבי (aggregate disability rate)
# שים לב: אם קיימת חפיפה בין קטגוריות יש צורך לבצע de-duplication (רק אם יש מזהה יחידני).
# כאן נעשה אגרגציה פשוטה אך שמרנית: ניקח max בין סכום לבין כלל, כדי לא להקטין.
# ---------------------------------------------------------------------------------
# אופציה א: aggregate_by_sum = sum של כל הקצבאות (אם אין כפל)
df['agg_count_sum'] = 0
for c in benefit_cols:
    if c in df.columns:
        df['agg_count_sum'] = df['agg_count_sum'] + df[c].fillna(0)
df['agg_rate_sum_per1000'] = df['agg_count_sum'] / df[COL_TOTAL_POP] * 1000

# אופציה ב (שמרנית): השתמש ב-general כייצוג בסיסי, אלא אם סה"כ גדול יותר
df['agg_rate_per1000'] = df['agg_rate_sum_per1000']
if COL_GENERAL in df.columns:
    df.loc[df['agg_rate_sum_per1000'] < (df[COL_GENERAL+'_per1000'].fillna(0)), 'agg_rate_per1000'] = df[COL_GENERAL+'_per1000']

# ---------------------------------------------------------------------------------
# 3) הגדרת "מוגבלויות הקשורות לעוני" (non-age, non-accident)
# ---------------------------------------------------------------------------------
# ניסוח בסיסי:
# poverty_related_count = general - longterm - work_injury
# אם יש שירותים מיוחדים שנחשבים לעוני (לדוגמה: special_services_for_persons_with_disabilities) תוכל לכלול
LONGTERM = COL_LONGTERM in df.columns
WORKINJ = COL_WORK_INJ in df.columns

df['poverty_related_count'] = df[COL_GENERAL].fillna(0)
if LONGTERM:
    df['poverty_related_count'] = df['poverty_related_count'] - df[COL_LONGTERM].fillna(0)
if WORKINJ:
    df['poverty_related_count'] = df['poverty_related_count'] - df[COL_WORK_INJ].fillna(0)
# לא לרדת מתחת לאפס
df['poverty_related_count'] = df['poverty_related_count'].clip(lower=0)
df['poverty_related_rate_per1000'] = df['poverty_related_count'] / df[COL_TOTAL_POP] * 1000

# ---------------------------------------------------------------------------------
# 4) משתנים מתערבים מומלצים לחסימה
# ---------------------------------------------------------------------------------
# תמיד לכלול: log(pop), %65+, peripherality, אם יש - unemployment, income_support etc.
df['log_pop'] = np.log(df[COL_TOTAL_POP])
if COL_POP_65 in df.columns:
    df['pct_65_plus'] = df[COL_POP_65] / df[COL_TOTAL_POP] * 100
else:
    # אם אין 65+, נסה לחשב מ־0_17 ו־18_64 אם יש
    if (COL_POP_0_17 in df.columns) and (COL_POP_18_64 in df.columns):
        df['population_65_plus'] = df[COL_TOTAL_POP] - df[COL_POP_0_17] - df[COL_POP_18_64]
        df['pct_65_plus'] = df['population_65_plus'] / df[COL_TOTAL_POP] * 100

# אם יש מדדי תעסוקה/תמיכה כלכלית
# COL_INCOME_SUPPORT = 'income_support_benefit'  # אם יש, המר והוסף ל-list של Confounders

# ---------------------------------------------------------------------------------
# 5) בדיקות קורלציה בסיסיות: Pearson + Spearman בין SEI ל־poverty_related_rate_per1000
# ומדידה של קורלציה חלקית תוך שליטה בלוג־אוכלוסייה ו-%65+
# ---------------------------------------------------------------------------------
target = 'poverty_related_rate_per1000'
if target not in df.columns or COL_SEI not in df.columns:
    raise ValueError("חסרים העמודות target או SEI. בדוק שמות העמודות והתאמן אותם בקוד.")

# Ensure target and control columns are numeric (float64) for statsmodels
df[target] = df[target].astype(float)
df[COL_SEI] = df[COL_SEI].astype(float)

# Pearson & Spearman (pairwise)
x = df[COL_SEI].dropna()
y = df.loc[x.index, target]
pearson_r, pearson_p = stats.pearsonr(x, y)
spearman_r, spearman_p = stats.spearmanr(x, y)

print("Pearson r (SEI vs poverty-related rate): r={:.3f}, p={:.3e}".format(pearson_r, pearson_p))
print("Spearman rho: rho={:.3f}, p={:.3e}".format(spearman_r, spearman_p))

# Partial correlation controlling for log_pop and pct_65_plus (regress-out residuals)
controls = ['log_pop']
if 'pct_65_plus' in df.columns:
    controls.append('pct_65_plus')

# make sure control cols exist and no NaN problem and convert to float
control_df = df[controls].astype(float)
valid_idx = df[[COL_SEI, target]].dropna().index.intersection(control_df.dropna().index)

res_y = sm.OLS(df.loc[valid_idx, target], sm.add_constant(control_df.loc[valid_idx, controls])).fit().resid
res_x = sm.OLS(df.loc[valid_idx, COL_SEI], sm.add_constant(control_df.loc[valid_idx, controls])).fit().resid
partial_r, partial_p = stats.pearsonr(res_x, res_y)
print("Partial correlation controlling for {}: r={:.3f}, p={:.3e}".format(controls, partial_r, partial_p))

# ---------------------------------------------------------------------------------
# 6) רגרסיה מרובת משתנים לחיזוק המסקנות (robust SE)
# ---------------------------------------------------------------------------------
predictors = [COL_SEI, 'log_pop']
if 'pct_65_plus' in df.columns:
    predictors.append('pct_65_plus')
if COL_PERIPH in df.columns:
    predictors.append(COL_PERIPH)

# Ensure predictor columns are numeric (float64)
X_reg = df[predictors].astype(float)

X = X_reg.dropna()
y_reg = df.loc[X.index, target].astype(float)
Xc = sm.add_constant(X)
model = sm.OLS(y_reg, Xc).fit(cov_type='HC3')  # robust SE
print(model.summary())

# תוספת: הפעל אינטראקציה SEI x peripherality אם שניהם קיימים
if (COL_SEI in df.columns) and (COL_PERIPH in df.columns):
    df['SEI_x_PER'] = df[COL_SEI] * df[COL_PERIPH]
    preds = [COL_SEI, COL_PERIPH, 'SEI_x_PER', 'log_pop']
    if 'pct_65_plus' in df.columns: preds.append('pct_65_plus')

    X2_reg = df[preds].astype(float)
    X2 = X2_reg.dropna()
    y2 = df.loc[X2.index, target].astype(float)
    X2c = sm.add_constant(X2)
    model2 = sm.OLS(y2, X2c).fit(cov_type='HC3')
    print("\nInteraction model summary (SEI x Peripherality):\n")
    print(model2.summary())

# ---------------------------------------------------------------------------------
# 7) זיהוי חריגים (יישובים שבהם observed >> predicted)
# ---------------------------------------------------------------------------------
# השתמש במודל האחרון שיצרת (או במודל בסיסי) כדי לחשב שאריות
if 'model' in locals():
    df.loc[X.index, 'predicted'] = model.predict(Xc)
    df['residual'] = df[target] - df['predicted']
    df['residual_std'] = (df['residual'] - df['residual'].mean()) / df['residual'].std()
    outliers = df.sort_values('residual', ascending=False).head(30)[['settlement_name', COL_TOTAL_POP, target, 'predicted','residual','residual_std']]
    print("\nTop positive residual outliers (likely institutional or hotspots):")
    display(outliers)

# ---------------------------------------------------------------------------------
# 8) גרפים להמחשה (scatter + residuals histogram)
# ---------------------------------------------------------------------------------

# Scatter plot: SEI vs poverty-related rate
plot_df = df.loc[X.index].copy() # Use the same filtered data as the regression model
# Ensure relevant columns are numeric for plotly.express color/size
plot_df['socio_economic_index_cluster'] = pd.to_numeric(plot_df['socio_economic_index_cluster'], errors='coerce')
plot_df['total_population'] = pd.to_numeric(plot_df['total_population'], errors='coerce')

fig_scatter = px.scatter(
    plot_df.dropna(subset=['socio_economic_index_score', 'poverty_related_rate_per1000', 'socio_economic_index_cluster', 'total_population']),
    x='socio_economic_index_score',
    y='poverty_related_rate_per1000',
    hover_name='settlement_name',
    size='total_population',
    size_max=60,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis', # Or any other suitable scale
    trendline='ols',
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'poverty_related_rate_per1000': 'Poverty-Related Disability Rate (per 1000)',
        'total_population': 'Population size',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Socio-Economic Index vs. Poverty-Related Disability Rate'
)
fig_scatter.update_layout(title_font_size=18, xaxis_title_font_size=14, yaxis_title_font_size=14)
fig_scatter.show()

# Residuals histogram
if 'residual' in df.columns:
    fig_hist = px.histogram(
        df.loc[X.index].dropna(subset=['residual']), # Use filtered data with residuals
        x='residual',
        nbins=40,
        title='Distribution of Residuals (Observed - Predicted)'
    )
    fig_hist.update_layout(title_font_size=18, xaxis_title_font_size=14, yaxis_title_font_size=14)
    fig_hist.show()

In [ ]:
df_senior = data_master.copy()

cols_for_plot = [
    'long_term_care_rate',
    'socio_economic_index_score',
    'population_65_plus',
    'settlement_name',
    'socio_economic_index_cluster'
]

# Ensure relevant columns are numeric, coercing errors to NaN
# Then explicitly convert ALL to standard float types (float64)
for col in ['long_term_care_rate', 'socio_economic_index_score', 'population_65_plus', 'socio_economic_index_cluster']:
    df_senior[col] = pd.to_numeric(df_senior[col], errors='coerce').astype(float)

# Drop rows with NaNs in any of the columns critical for plotting
# Now that they are all float64, dropna will work cleanly
df_senior = df_senior.dropna(subset=['long_term_care_rate', 'socio_economic_index_score', 'population_65_plus', 'socio_economic_index_cluster'])

fig = px.scatter(
    df_senior,
    x='socio_economic_index_score',
    y='long_term_care_rate',
    hover_name='settlement_name',
    size='population_65_plus',
    size_max=60,
    color='socio_economic_index_cluster',
    color_continuous_scale='Viridis',
    trendline='ols',
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'long_term_care_rate': 'Long-Term Care Rate (per 100 seniors)',
        'population_65_plus': 'Population 65+',
        'socio_economic_index_cluster': 'Socio-Economic Index Cluster'
    },
    title='Correlation: Senior Long-Term Care Rate vs. Socio-Economic Index',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

In [ ]:
import plotly.express as px

COL_SEI = 'socio_economic_index_score'
COL_WORK_INJ = "work_injury_victims_receiving_disability_and_dependents’_benefits"
COL_TOTAL_POP = 'total_population'
COL_SEI_CLUSTER = 'socio_economic_index_cluster'

df_plot = data_master.copy()

# Ensure relevant columns are numeric and handle NaNs for plotting
for col in [COL_SEI, COL_WORK_INJ, COL_TOTAL_POP, COL_SEI_CLUSTER]:
    df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce').astype(float)

# Calculate work_injury_rate_per_1000 *before* comprehensive filtering
df_plot['work_injury_rate_per1000'] = (
    df_plot[COL_WORK_INJ] / df_plot[COL_TOTAL_POP] * 1000
)

# Filter out rows with missing values after conversion, and ensure total_population is positive
# Also filter on the newly created rate column
df_plot = df_plot[
    df_plot[COL_SEI].notna() &
    df_plot[COL_WORK_INJ].notna() & # Keep this if COL_WORK_INJ itself is still needed for context
    df_plot[COL_TOTAL_POP].notna() &
    (df_plot[COL_TOTAL_POP] > 0) &
    df_plot['work_injury_rate_per1000'].notna() # Ensure the rate itself is not NaN
].copy()

# Create the interactive scatter plot
fig = px.scatter(
    df_plot,
    x=COL_SEI,
    y='work_injury_rate_per1000',
    hover_name='settlement_name',
    size=COL_TOTAL_POP,
    size_max=60, # Max size for the bubbles
    color=COL_SEI_CLUSTER,
    color_continuous_scale='Magma', # Changed color scale for variety
    trendline='ols',
    labels={
        COL_SEI: 'Socio-Economic Index Score',
        'work_injury_rate_per1000': 'Work Injury Disability Rate (per 1,000)',
        COL_TOTAL_POP: 'Total Population',
        COL_SEI_CLUSTER: 'Socio-Economic Index Cluster'
    },
    title='Socio-Economic Index vs. Work-Injury Disability Rate',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

# Recalculate correlations on the cleaned data for consistency with the plot
x_corr = df_plot[COL_SEI]
y_corr = df_plot['work_injury_rate_per1000']

pearson_r, pearson_p = stats.pearsonr(x_corr, y_corr)
spearman_r, spearman_p = stats.spearmanr(x_corr, y_corr)

print(f"Pearson r: {pearson_r:.3f} (p={pearson_p:.3e})")
print(f"Spearman rho: {spearman_r:.3f} (p={spearman_p:.3e})")

# Task
The user wants to identify and visualize 'bad' and 'good' outlier settlements based on their socio-economic index score and poverty-related disability rate.

To achieve this, I will perform the following steps:
1.  **Prepare Data for Outlier Analysis**: I will create a new DataFrame `df_outliers` from `data_master`, including `socio_economic_index_score`, `poverty_related_rate_per1000`, `socio_economic_index_cluster`, `settlement_name`, and `total_population`. I will ensure all numeric columns are correctly typed and handle any missing values by dropping rows where these key metrics are NaN, which is crucial for accurate percentile calculations and outlier identification.
2.  **Define Outlier Thresholds**: I will calculate the 25th (Q1) and 75th (Q3) percentiles for `socio_economic_index_score` and `poverty_related_rate_per1000`. These percentiles will serve as thresholds to classify settlements.
3.  **Identify and Classify Outliers**: I will create a new column, `Outlier_Type`, in `df_outliers`.
    *   A settlement will be classified as a **'Bad Outlier'** if it has a low `socio_economic_index_score` (below Q1) AND a high `poverty_related_rate_per1000` (above Q3). These represent highly disadvantaged settlements with a high burden of poverty-related disability.
    *   A settlement will be classified as a **'Good Outlier'** if it has a low `socio_economic_index_score` (below Q1) AND a low `poverty_related_rate_per1000` (below Q1). These represent disadvantaged settlements that are performing relatively well in terms of poverty-related disability rates.
    *   All other settlements will be labeled **'Normal'**.
4.  **Examine Outlier Settlements**: I will display the top 'Bad Outlier' and 'Good Outlier' settlements, showing their `settlement_name`, `socio_economic_index_score`, `poverty_related_rate_per1000`, `socio_economic_index_cluster`, and `total_population`.
5.  **Visualize Outliers**: I will generate an interactive scatter plot using `plotly.express`. The x-axis will represent `socio_economic_index_score` and the y-axis `poverty_related_rate_per1000`. Points will be colored based on their `Outlier_Type` (red for 'Bad Outlier', green for 'Good Outlier', blue for 'Normal'), and their size will reflect `total_population` for additional context. Hover functionality will display settlement details.
6.  **Final Task**: I will summarize the characteristics of the identified outlier settlements and discuss the insights gained from their classification and visualization, addressing the initial query about understanding 'bad' and 'good' outliers and their comparison within socio-economic clusters.

This comprehensive approach will allow us to pinpoint settlements that deviate significantly from expected patterns based on their socio-economic standing and disability rates.

## Prepare Data for Outlier Analysis

### Subtask:
Ensure 'socio_economic_index_score', 'poverty_related_rate_per1000', 'socio_economic_index_cluster', 'settlement_name', and 'total_population' columns in `data_master` are numeric and handle any missing values, preparing the data for outlier identification.


**Reasoning**:
The subtask requires several data preparation steps: calculating new rates, creating a working DataFrame copy, converting specified columns to numeric types, and handling missing values in those columns. Combining these into a single code block ensures efficient data manipulation.



In [ ]:
data_master['poverty_related_disability'] = (
    data_master['general_disability_benefit'].fillna(0) +
    data_master['special_services_for_persons_with_disabilities'].fillna(0)
)

data_master['poverty_related_rate_per1000'] = (
    data_master['poverty_related_disability'] /
    data_master['population_18_64']
) * 1000

df_outliers = data_master.copy()

cols_for_outliers = [
    'socio_economic_index_score',
    'poverty_related_rate_per1000',
    'socio_economic_index_cluster',
    'settlement_name',
    'total_population'
]

for col in cols_for_outliers:
    if col != 'settlement_name': # settlement_name is string, no need to convert to numeric
        df_outliers[col] = pd.to_numeric(df_outliers[col], errors='coerce')

df_outliers = df_outliers.dropna(subset=cols_for_outliers).copy()

## Define Outlier Thresholds

### Subtask:
Calculate the 25th and 75th percentiles for 'socio_economic_index_score' and 'poverty_related_rate_per1000' to establish criteria for 'low'/'high' values for outlier identification.


**Reasoning**:
To define outlier thresholds, I will calculate the 25th and 75th percentiles for 'socio_economic_index_score' and 'poverty_related_rate_per1000' from the `df_outliers` DataFrame.



In [ ]:
sei_q1 = df_outliers['socio_economic_index_score'].quantile(0.25)
sei_q3 = df_outliers['socio_economic_index_score'].quantile(0.75)
poverty_rate_q1 = df_outliers['poverty_related_rate_per1000'].quantile(0.25)
poverty_rate_q3 = df_outliers['poverty_related_rate_per1000'].quantile(0.75)

print(f"Socio-Economic Index Score Q1: {sei_q1:.2f}")
print(f"Socio-Economic Index Score Q3: {sei_q3:.2f}")
print(f"Poverty-Related Rate per 1000 Q1: {poverty_rate_q1:.2f}")
print(f"Poverty-Related Rate per 1000 Q3: {poverty_rate_q3:.2f}")

## Identify and Classify Outliers

### Subtask:
Create a new column 'Outlier_Type' in `df_outliers`. Classify settlements as 'Bad Outlier' (low SEI score and high disability-to-poverty rate), 'Good Outlier' (low SEI score and low disability-to-poverty rate), or 'Normal' based on the defined thresholds.


**Reasoning**:
I will classify settlements into 'Bad Outlier', 'Good Outlier', or 'Normal' categories in the 'Outlier_Type' column of `df_outliers` based on the specified conditions and previously calculated quartile thresholds.



In [ ]:
df_outliers['Outlier_Type'] = 'Normal'
df_outliers.loc[
    (df_outliers['socio_economic_index_score'] < sei_q1) &
    (df_outliers['poverty_related_rate_per1000'] > poverty_rate_q3),
    'Outlier_Type'
] = 'Bad Outlier'
df_outliers.loc[
    (df_outliers['socio_economic_index_score'] < sei_q1) &
    (df_outliers['poverty_related_rate_per1000'] < poverty_rate_q1),
    'Outlier_Type'
] = 'Good Outlier'

print("Settlements classified into Outlier_Type column.")

## Examine Outlier Settlements

### Subtask:
Display the top 'Bad Outlier' and 'Good Outlier' settlements, showing their `settlement_name`, `socio_economic_index_score`, `poverty_related_rate_per1000`, `socio_economic_index_cluster`, and `total_population`.



**Reasoning**:
To examine the outlier settlements, I will filter `df_outliers` for 'Bad Outlier' and 'Good Outlier' types, sort them by relevant metrics, and display the top 10 settlements for each category with the requested columns.



In [ ]:
print("Top 10 Bad Outlier Settlements:")
df_bad_outliers = df_outliers[df_outliers['Outlier_Type'] == 'Bad Outlier'].sort_values(
    by=['socio_economic_index_score', 'poverty_related_rate_per1000'], ascending=[True, False]
).head(10)
display(df_bad_outliers[cols_for_outliers])

print("\nTop 10 Good Outlier Settlements:")
df_good_outliers = df_outliers[df_outliers['Outlier_Type'] == 'Good Outlier'].sort_values(
    by=['socio_economic_index_score', 'poverty_related_rate_per1000'], ascending=[True, True]
).head(10)
display(df_good_outliers[cols_for_outliers])

**Reasoning**:
Now that the outlier settlements have been identified and examined, I will create an interactive scatter plot to visualize the 'Bad Outlier', 'Good Outlier', and 'Normal' settlements based on their socio-economic index score and poverty-related disability rate, as described in the overall task.



In [ ]:
color_map = {
    'Bad Outlier': 'red',
    'Good Outlier': 'green',
    'Normal': 'blue'
}

fig = px.scatter(
    df_outliers,
    x='socio_economic_index_score',
    y='poverty_related_rate_per1000',
    hover_name='settlement_name',
    size='total_population',
    size_max=60,
    color='Outlier_Type',
    color_discrete_map=color_map,
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'poverty_related_rate_per1000': 'Poverty-Related Disability Rate (per 1,000)',
        'total_population': 'Population size',
        'Outlier_Type': 'Outlier Classification'
    },
    title='Socio-Economic Index vs. Poverty-Related Disability Rate with Outliers',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

## Summary of Outlier Settlements

The analysis successfully identified settlements that are 'Bad Outliers' and 'Good Outliers' based on their socio-economic index score and poverty-related disability rates.

### Bad Outliers:
These settlements are characterized by a low socio-economic index score (indicating disadvantage) and a high poverty-related disability rate. The displayed 'Bad Outliers' typically show:
- **Low Socio-Economic Index Score**: Consistently below the 25th percentile (`sei_q1`).
- **High Poverty-Related Disability Rate**: Consistently above the 75th percentile (`poverty_rate_q3`).
- **Demographics**: Many of these settlements are smaller to medium-sized, often with lower socio-economic clusters (e.g., cluster 2 or 3).

These settlements represent communities facing significant dual challenges: deep socio-economic disadvantage coupled with a disproportionately high burden of poverty-related disabilities. This suggests a potential vicious cycle where poverty exacerbates disability, and disability perpetuates poverty within these communities. They could be targets for comprehensive interventions aimed at both socio-economic upliftment and disability support.

### Good Outliers:
These settlements are also characterized by a low socio-economic index score (indicating disadvantage), but they exhibit a surprisingly low poverty-related disability rate. The displayed 'Good Outliers' typically show:
- **Low Socio-Economic Index Score**: Consistently below the 25th percentile (`sei_q1`).
- **Low Poverty-Related Disability Rate**: Consistently below the 25th percentile (`poverty_rate_q1`).
- **Demographics**: These can vary, but like 'Bad Outliers', they tend to be on the lower end of the socio-economic cluster scale.

'Good Outliers' are particularly interesting. Despite facing similar socio-economic challenges as 'Bad Outliers', they manage to maintain lower rates of poverty-related disability. This raises important questions about what factors contribute to their relative resilience. These could include:
- **Effective Local Programs**: Local initiatives or community support systems that mitigate the impact of poverty on disability.
- **Demographic Nuances**: Specific demographic compositions (e.g., strong family support structures, cultural factors) that are not captured by the general socio-economic index.
- **Data Limitations**: Potential underreporting of disabilities or differences in access to benefits.

Studying these 'Good Outliers' could provide valuable insights into protective factors and successful strategies that could potentially be replicated in 'Bad Outlier' communities.

### Insights from Visualization:
The scatter plot visually reinforces these classifications. 'Bad Outliers' cluster in the bottom-right (low SEI score, high disability rate), while 'Good Outliers' appear in the bottom-left (low SEI score, low disability rate). The majority of 'Normal' settlements fall within the expected range, generally showing an inverse relationship between socio-economic standing and disability rates, with better-off communities (higher SEI score) tending to have lower poverty-related disability rates.

This analysis highlights the importance of looking beyond just overall socio-economic indicators and delving into specific outcomes like disability rates to identify communities that are either struggling more than expected or performing surprisingly well, thus informing targeted policy interventions.

## Final Task

### Subtask:
Summarize the characteristics of the identified outlier settlements and discuss the insights gained from their classification and visualization, addressing the initial query about understanding 'bad' and 'good' outliers and their comparison within socio-economic clusters.


## Summary:

### Q&A

**What constitutes a 'bad' outlier settlement?**
A 'bad' outlier settlement is defined as one with a low socio-economic index score (below the 25th percentile, which is -0.78) and a high poverty-related disability rate (above the 75th percentile, which is 99.74 per 1,000 population). These settlements face significant dual challenges: deep socio-economic disadvantage combined with a disproportionately high burden of poverty-related disabilities.

**What constitutes a 'good' outlier settlement?**
A 'good' outlier settlement is defined as one with a low socio-economic index score (below the 25th percentile, which is -0.78) but an unexpectedly low poverty-related disability rate (below the 25th percentile, which is 51.52 per 1,000 population). These settlements demonstrate a degree of resilience, managing to keep poverty-related disability rates low despite their socio-economic challenges.

**How do 'bad' and 'good' outliers compare within socio-economic clusters?**
Both 'bad' and 'good' outliers tend to be found within the lower socio-economic clusters (e.g., cluster 2 or 3), indicating that both types of outliers originate from disadvantaged communities. The key difference lies in their poverty-related disability rates, which are significantly divergent despite similar socio-economic starting points.

### Data Analysis Key Findings

*   **Outlier Definition Thresholds**: The 25th percentile (Q1) for the socio-economic index score was -0.78, and for the poverty-related disability rate was 51.52 per 1,000. The 75th percentile (Q3) for the socio-economic index score was 0.73, and for the poverty-related disability rate was 99.74 per 1,000. These thresholds were used to classify settlements.
*   **'Bad Outlier' Characteristics**: Identified 'Bad Outliers' like 'כפר מנדא' and 'ג'סר א-זרקא' typically exhibited socio-economic index scores around -1.45 and poverty-related disability rates exceeding 100.56 per 1,000. These settlements are often smaller to medium-sized and fall into lower socio-economic clusters.
*   **'Good Outlier' Characteristics**: Identified 'Good Outliers' such as 'קצר א-סר' and 'קריית יערים' displayed socio-economic index scores as low as -2.41 but maintained surprisingly low poverty-related disability rates, around 48.09 per 1,000. These also tend to be in lower socio-economic clusters.
*   **Visualization Confirmation**: An interactive scatter plot visually confirmed that 'Bad Outliers' clustered in the bottom-right of the plot (low socio-economic index, high disability rate), while 'Good Outliers' were situated in the bottom-left (low socio-economic index, low disability rate), with 'Normal' settlements filling the central distribution.

### Insights or Next Steps

*   **Targeted Intervention for 'Bad Outliers'**: 'Bad Outlier' settlements represent critical areas for comprehensive, integrated interventions addressing both socio-economic upliftment and enhanced disability support, as they face compounding disadvantages.
*   **Learn from 'Good Outliers'**: Further qualitative and quantitative research into 'Good Outlier' settlements could uncover protective factors or successful local strategies (e.g., effective community programs, unique demographic resilience) that enable lower poverty-related disability rates despite overall disadvantage, offering replicable models for other communities.


# Task
**Re-initialize DataMaster and Rates**: Execute the original cells to ensure `data_master` is correctly loaded, processed, and all rate-related columns (including `long_term_care_rate`) are calculated and available in the kernel.
**Prepare Data for Senior Outlier Analysis**: Create a new DataFrame, `df_senior_outliers`, from `data_master`. Ensure 'long_term_care_rate', 'socio_economic_index_score', 'socio_economic_index_cluster', 'settlement_name', and 'population_65_plus' columns are numeric and handle any missing values by dropping rows where these key metrics are NaN. This step will also calculate the long_term_care_rate if it hasn't been done already.
**Define Senior Outlier Thresholds**: Calculate the 25th (Q1) and 75th (Q3) percentiles for 'socio_economic_index_score' and 'long_term_care_rate' from `df_senior_outliers`. These quartiles will be used to define the 'high-medium' SEI range and 'high'/'low' care rate thresholds.
**Identify and Classify Senior Outliers**: Create a new column 'Outlier_Type_Senior' in `df_senior_outliers`. Classify settlements as:
- **'High Outlier'**: If `socio_economic_index_score` is greater than Q1 AND `long_term_care_rate` is greater than Q3.
- **'Low Outlier'**: If `socio_economic_index_score` is less than Q1 AND `long_term_care_rate` is less than Q1.
- **'Normal'**: For all other settlements.
**Examine Senior Outlier Settlements**: Display the top 'High Outlier' and 'Low Outlier' settlements. Show their 'settlement_name', 'socio_economic_index_score', 'long_term_care_rate', 'socio_economic_index_cluster', and 'population_65_plus'. This will allow for comparison within SEI clusters and identification of the most prominent outliers.
**Visualize Senior Outliers**: Generate an interactive scatter plot using `plotly.express`. The x-axis will represent 'socio_economic_index_score' and the y-axis 'long_term_care_rate'. Points will be colored based on their 'Outlier_Type_Senior' (e.g., distinct colors for 'High Outlier', 'Low Outlier', and 'Normal'), and their size will reflect 'population_65_plus' for additional context. Hover functionality will display settlement details.
**Final Task**: Summarize the characteristics of the identified 'High' and 'Low' outlier settlements in terms of Senior Long-Term Care Rate and Socio-Economic Index. Discuss the insights gained from their classification and visualization, addressing their comparison within SEI clusters and the factors that might contribute to their outlier status.

## Re-initialize DataMaster and Rates

### Subtask:
Ensure `data_master` is correctly loaded, processed, and all rate-related columns (including `long_term_care_rate`) are calculated and available in the kernel.


**Reasoning**:
The subtask requires re-initializing `data_master` and ensuring rate-related columns are calculated. The first step is to run the cell that loads, cleans, and merges the initial datasets into `data_master`.



In [ ]:
paths = {
    "benefits": "benefits_2024_12.xlsx",
    "lamas": "p_libud_23.xlsx",
    "socio_regional": "24_24_230t3.xlsx",
    "periph_regional": "24_22_420t3.xlsx",
    "coordinates": "israel_settlements_all_with_coords.csv"
}

dfs = load_data(paths)

df_benefits = dfs["benefits"].iloc[5:].copy()
df_benefits = df_benefits.reset_index(drop=True)
df_benefits.columns = [
    "settlement_name",
    "settlement_symbol",
    "settlement_type",
    "total_population",
    "population_0_17",
    "population_18_64",
    "population_65_plus",
    "total_recipients_of_old_age_and/or_survivors’_benefits",
    "recipients_of_the_pension_with_income_supplementation",
    "recipients_of_the_senior_citizen_pension_only",
    "long_term_care_benefit",
    "general_disability_benefit",
    "special_services_for_persons_with_disabilities",
    "disabled_child_benefit",
    "mobility_benefit",
    "work_injury_victims_receiving_disability_and_dependents’_benefits",
    "injury_allowance",
    "num_families_receiving_child_benefit",
    "num_children_receiving_child_benefit",
    "families_with_4+_children_receiving_child_benefit",
    "maternity_benefits",
    "alimony",
    "income_support_benefit",
    "unemployment_benefit"
]

df_lamas = dfs["lamas"].iloc[9:].copy()
df_lamas = df_lamas.reset_index(drop=True)
df_lamas = df_lamas[df_lamas[3] != 'מועצה אזורית']
df_lamas.rename(columns={1: "settlement_symbol", 250: "socio_economic_index_cluster",
                         251: "socio_economic_index_score",256: "peripherality_index_cluster",
                         257: "peripherality_index_score"}, inplace=True)

df_socio = dfs["socio_regional"].iloc[10:].copy()
df_socio = df_socio.reset_index(drop=True)
df_socio = df_socio.iloc[:-8]
df_socio.rename(columns={5: "settlement_symbol", 12: "socio_economic_index_cluster", 10: "socio_economic_index_score"}, inplace=True)

#DROP_ON_TOP=9
df_periph = dfs["periph_regional"].iloc[9:].copy()
#NOTES_IN_THE_END=4
df_periph = df_periph.reset_index(drop=True)
df_periph = df_periph.iloc[:-4]
df_periph.rename(columns={4: "settlement_symbol", 12: "peripherality_index_cluster", 10: "peripherality_index_score"}, inplace=True)

data_master = merge_lamas(df_benefits, df_lamas)
data_master = merge_index_from_regional(data_master, df_socio,
    index_cols=[
        "socio_economic_index_cluster",
        "socio_economic_index_score"
    ]
)
data_master = merge_index_from_regional(data_master, df_periph,
    index_cols=[
        "peripherality_index_cluster",
        "peripherality_index_score"
    ]
)
data_master = merge_coordinates(data_master, dfs["coordinates"])
data_master = clean_values(data_master)

In [ ]:
df_senior_outliers = data_master.copy()

# Ensure long_term_care_rate is calculated
df_senior_outliers['long_term_care_rate'] = round(
    (df_senior_outliers['long_term_care_benefit'] / df_senior_outliers['population_65_plus']) * 100,
    2
)

cols_for_senior_outliers = [
    'long_term_care_rate',
    'socio_economic_index_score',
    'socio_economic_index_cluster',
    'settlement_name',
    'population_65_plus'
]

# Ensure relevant columns are numeric, coercing errors to NaN
# Then explicitly convert ALL to standard float types (float64)
for col in cols_for_senior_outliers:
    if col != 'settlement_name': # settlement_name is a string and does not need to be converted to numeric
        df_senior_outliers[col] = pd.to_numeric(df_senior_outliers[col], errors='coerce').astype(float)

# Drop rows with NaNs in any of the columns critical for analysis
df_senior_outliers = df_senior_outliers.dropna(subset=cols_for_senior_outliers).copy()

print(f"Prepared df_senior_outliers with {len(df_senior_outliers)} rows.")

In [ ]:
sei_q1_senior = df_senior_outliers['socio_economic_index_score'].quantile(0.25)
sei_q3_senior = df_senior_outliers['socio_economic_index_score'].quantile(0.75)
long_term_care_rate_q1 = df_senior_outliers['long_term_care_rate'].quantile(0.25)
long_term_care_rate_q3 = df_senior_outliers['long_term_care_rate'].quantile(0.75)

print(f"Socio-Economic Index Score Q1 for Seniors: {sei_q1_senior:.2f}")
print(f"Socio-Economic Index Score Q3 for Seniors: {sei_q3_senior:.2f}")
print(f"Long-Term Care Rate Q1: {long_term_care_rate_q1:.2f}")
print(f"Long-Term Care Rate Q3: {long_term_care_rate_q3:.2f}")

In [ ]:
df_senior_outliers['Outlier_Type_Senior'] = 'Normal'
df_senior_outliers.loc[
    (df_senior_outliers['socio_economic_index_score'] > sei_q1_senior) &
    (df_senior_outliers['long_term_care_rate'] > long_term_care_rate_q3),
    'Outlier_Type_Senior'
] = 'High Outlier'
df_senior_outliers.loc[
    (df_senior_outliers['socio_economic_index_score'] < sei_q1_senior) &
    (df_senior_outliers['long_term_care_rate'] < long_term_care_rate_q1),
    'Outlier_Type_Senior'
] = 'Low Outlier'

print("Settlements classified into Outlier_Type_Senior column.")

In [ ]:
print("Top 10 High Outlier Settlements:")
df_high_outliers = df_senior_outliers[df_senior_outliers['Outlier_Type_Senior'] == 'High Outlier'].sort_values(
    by=['socio_economic_index_score', 'long_term_care_rate'], ascending=[True, False]
).head(10)
display(df_high_outliers[cols_for_senior_outliers])

print("\nTop 10 Low Outlier Settlements:")
df_low_outliers = df_senior_outliers[df_senior_outliers['Outlier_Type_Senior'] == 'Low Outlier'].sort_values(
    by=['socio_economic_index_score', 'long_term_care_rate'], ascending=[True, True]
).head(10)
display(df_low_outliers[cols_for_senior_outliers])

In [ ]:
color_map_senior = {
    'High Outlier': 'red',
    'Low Outlier': 'green',
    'Normal': 'blue'
}

fig = px.scatter(
    df_senior_outliers,
    x='socio_economic_index_score',
    y='long_term_care_rate',
    hover_name='settlement_name',
    size='population_65_plus',
    size_max=60,
    color='Outlier_Type_Senior',
    color_discrete_map=color_map_senior,
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'long_term_care_rate': 'Long-Term Care Rate (per 100 seniors)',
        'population_65_plus': 'Population 65+',
        'Outlier_Type_Senior': 'Senior Outlier Classification'
    },
    title='Socio-Economic Index vs. Senior Long-Term Care Rate with Outliers',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()

In [ ]:
df_work_injury_outliers = data_master.copy()

cols_for_work_injury_outliers = [
    'work_injury_victims_rate',
    'socio_economic_index_score',
    'socio_economic_index_cluster',
    'settlement_name',
    'total_population'
]

for col in cols_for_work_injury_outliers:
    if col != 'settlement_name': # settlement_name is string, no need to convert to numeric
        df_work_injury_outliers[col] = pd.to_numeric(df_work_injury_outliers[col], errors='coerce')

df_work_injury_outliers = df_work_injury_outliers.dropna(subset=cols_for_work_injury_outliers).copy()

print("df_work_injury_outliers created and cleaned.")

In [ ]:
sei_q1_work_injury = df_work_injury_outliers['socio_economic_index_score'].quantile(0.25)
sei_q3_work_injury = df_work_injury_outliers['socio_economic_index_score'].quantile(0.75)
work_injury_rate_q1 = df_work_injury_outliers['work_injury_victims_rate'].quantile(0.25)
work_injury_rate_q3 = df_work_injury_outliers['work_injury_victims_rate'].quantile(0.75)

print(f"Socio-Economic Index Score Q1 for Work Injury: {sei_q1_work_injury:.2f}")
print(f"Socio-Economic Index Score Q3 for Work Injury: {sei_q3_work_injury:.2f}")
print(f"Work Injury Victims Rate Q1: {work_injury_rate_q1:.2f}")
print(f"Work Injury Victims Rate Q3: {work_injury_rate_q3:.2f}")

In [ ]:
df_work_injury_outliers['Outlier_Type_Work_Injury'] = 'Normal'
df_work_injury_outliers.loc[
    (df_work_injury_outliers['socio_economic_index_score'] > sei_q1_work_injury) &
    (df_work_injury_outliers['work_injury_victims_rate'] > work_injury_rate_q3),
    'Outlier_Type_Work_Injury'
] = 'High Outlier'
df_work_injury_outliers.loc[
    (df_work_injury_outliers['socio_economic_index_score'] < sei_q1_work_injury) &
    (df_work_injury_outliers['work_injury_victims_rate'] < work_injury_rate_q1),
    'Outlier_Type_Work_Injury'
] = 'Low Outlier'

print("Settlements classified into Outlier_Type_Work_Injury column.")

In [ ]:
print("Top 10 High Outlier Settlements:")
df_high_work_injury_outliers = df_work_injury_outliers[df_work_injury_outliers['Outlier_Type_Work_Injury'] == 'High Outlier'].sort_values(
    by=['socio_economic_index_score', 'work_injury_victims_rate'], ascending=[False, False] # High SEI and High rate
).head(10)
display(df_high_work_injury_outliers[cols_for_work_injury_outliers])

print("\nTop 10 Low Outlier Settlements:")
df_low_work_injury_outliers = df_work_injury_outliers[df_work_injury_outliers['Outlier_Type_Work_Injury'] == 'Low Outlier'].sort_values(
    by=['socio_economic_index_score', 'work_injury_victims_rate'], ascending=[True, True] # Low SEI and Low rate
).head(10)
display(df_low_work_injury_outliers[cols_for_work_injury_outliers])

In [ ]:
color_map_work_injury = {
    'High Outlier': 'red',
    'Low Outlier': 'green',
    'Normal': 'blue'
}

fig = px.scatter(
    df_work_injury_outliers,
    x='socio_economic_index_score',
    y='work_injury_victims_rate',
    hover_name='settlement_name',
    size='total_population',
    size_max=60,
    color='Outlier_Type_Work_Injury',
    color_discrete_map=color_map_work_injury,
    labels={
        'socio_economic_index_score': 'Socio-Economic Index Score',
        'work_injury_victims_rate': 'Work Injury Victims Rate (per 100)',
        'total_population': 'Total Population',
        'Outlier_Type_Work_Injury': 'Work Injury Outlier Classification'
    },
    title='Socio-Economic Index vs. Work Injury Victims Rate with Outliers',
    width=1400,
    height=750
)

fig.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig.show()